# 06 - SH + Context Reranking (CrossEncoder)

**Setup**: Schwartz-Hearst index + CrossEncoder reranking.

CR mengambil top-20 dari hybrid retrieval, lalu rerank menggunakan
`ms-marco-MiniLM-L-6-v2` untuk ambil top-5 final.

**Output**: `results/BM25_Expansion/sh_cr_openai_phase{1,2}.json`

**Estimasi**:
- Phase 1: ~17 menit (CrossEncoder ringan, CPU OK)
- Phase 2: ~25 menit
- Total: ~42 menit, ~$11


In [1]:
import os, sys, json, pickle, time, re, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
from pathlib import Path
from datetime import datetime

from openai import OpenAI
from rank_bm25 import BM25Okapi
from datasets import load_dataset
import chromadb

warnings.filterwarnings("ignore")


@dataclass
class Document:
    text: str; pubid: str; question: str
    section_label: str; answer: str; decision: str

@dataclass
class RetrievalResult:
    document: Document; score: float; doc_id: int = -1
    bm25_score: float = 0.0; dense_score: float = 0.0
    rrf_score: float = 0.0; reranker_score: float = 0.0

import __main__
__main__.Document = Document

# Set env var
# os.environ["OPENAI_API_KEY"] = "<REDACTED — set via shell env or .env file>"

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY belum di-set.")

print(f"Python: {sys.version.split()[0]} | NumPy: {np.__version__} | chromadb: {chromadb.__version__}")


C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: 3.11.9 | NumPy: 2.3.5 | chromadb: 1.5.8


In [ ]:
# ============================================================
# PATH SETUP — auto-resolve PROJECT_ROOT
# ============================================================
from pathlib import Path

_HERE = Path('.').resolve()
PROJECT_ROOT = next((p for p in [_HERE] + list(_HERE.parents) if p.name == 'Code TA'), _HERE.parent.parent.parent)
NOTEBOOKS_V2 = PROJECT_ROOT / 'notebooks'
INDEXES_DIR  = NOTEBOOKS_V2 / 'indexes'
RESULTS_DIR  = PROJECT_ROOT / 'results' / '30_hybrid_dengan_ekspansi'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

BM25_INDEX_PATH = INDEXES_DIR / 'pubmedqa_bm25_ekspansi.pkl'
CHROMA_DB_PATH  = INDEXES_DIR / 'pubmedqa_chroma_ekspansi'
NOTEBOOK_DIR    = INDEXES_DIR  # kompat lama: BM25_INDEX_PATH dan CHROMA_DB_PATH

print(f'PROJECT_ROOT  : {PROJECT_ROOT}')
print(f'INDEXES_DIR   : {INDEXES_DIR}')
print(f'RESULTS_DIR   : {RESULTS_DIR}')
print(f'BM25 index    : {BM25_INDEX_PATH.name}')
print(f'Chroma DB     : {CHROMA_DB_PATH.name}')


In [2]:
LLM_MODEL = "gpt-4.1-mini"
EMBED_MODEL = "text-embedding-3-small"
TOP_K_BM25 = 50; TOP_K_DENSE = 50; TOP_K_RETRIEVAL = 5
TOP_K_RERANKER = 20  # candidates before CrossEncoder rerank (only used if CR=True)

DATASET_NAME = "qiaojin/PubMedQA"; DATASET_SUBSET = "pqa_labeled"
MAX_SAMPLES = 500
TEMPERATURE = 0.0; SEED = 42
MAX_REWRITE_QUERIES = 3  # max queries from rewriter

HERE = Path(".").resolve()
NOTEBOOKS_DIR = HERE.parent if HERE.name == "BM25 Expansion" else HERE
PROJECT_ROOT = NOTEBOOKS_DIR.parent
# BM25_INDEX_PATH = NOTEBOOKS_DIR / "pubmedqa_bm25_sh.pkl"
# CHROMA_DB_PATH = NOTEBOOKS_DIR / "pubmedqa_chroma_sh"
# RESULTS_DIR = PROJECT_ROOT / "results" / "BM25_Expansion"
# RESULTS_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_NAME = "sh_cr_openai"
PHASE1_PATH = RESULTS_DIR / f"{CONFIG_NAME}_phase1_answers.json"
PHASE2_PATH = RESULTS_DIR / f"{CONFIG_NAME}_phase2_custom.json"

print("Konfigurasi:")
print(f"  LLM         : {LLM_MODEL}")
print(f"  Method      : CrossEncoder Rerank")
print(f"  BM25 index  : {BM25_INDEX_PATH.name} (Schwartz-Hearst)")
print(f"  Chroma path : {CHROMA_DB_PATH.name} (Schwartz-Hearst)")
print(f"  Sampel      : {MAX_SAMPLES}")
print(f"  Output      : {PHASE1_PATH}")

assert BM25_INDEX_PATH.exists(), f"BM25 SH tidak ditemukan. Run build_sh_index.py."
assert CHROMA_DB_PATH.exists(), f"Chroma SH tidak ditemukan. Run notebook 03 dulu."


Konfigurasi:
  LLM         : gpt-4.1-mini
  Method      : CrossEncoder Rerank
  BM25 index  : pubmedqa_bm25_sh.pkl (Schwartz-Hearst)
  Chroma path : pubmedqa_chroma_sh (Schwartz-Hearst)
  Sampel      : 500
  Output      : C:\Users\Ricky Wijaya\Documents\STI\Semester 8\TA\Code TA\results\BM25_Expansion\sh_cr_openai_phase1_answers.json


In [3]:
def tokenize_bm25(text):
    return re.sub(r"[^a-zA-Z0-9\s]", " ", text.lower()).split()

with open(BM25_INDEX_PATH, "rb") as f:
    saved = pickle.load(f)
bm25_index = saved["bm25"]; documents = saved["documents"]
print(f"Loaded BM25 SH: {len(documents)} chunks")

chroma_client = chromadb.PersistentClient(path=str(CHROMA_DB_PATH))
chroma_collection = chroma_client.get_collection(name="pubmedqa_docs_sh")
print(f"Loaded Chroma SH: {chroma_collection.count()} vectors")

ds = load_dataset(DATASET_NAME, DATASET_SUBSET)["train"]
pubmedqa_data = ds.select(range(MAX_SAMPLES))
print(f"Loaded {len(pubmedqa_data)} samples")


Loaded BM25 SH: 1706 chunks
Loaded Chroma SH: 1706 vectors


Loaded 500 samples


In [4]:
client = OpenAI(api_key=api_key)


def openai_generate(prompt, max_tokens=300, temperature=TEMPERATURE):
    for attempt in range(5):
        try:
            resp = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature, max_tokens=max_tokens, seed=SEED,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            err = str(e)
            if "429" in err or "rate" in err.lower():
                time.sleep((attempt + 1) * 10)
            elif "500" in err or "502" in err or "503" in err:
                time.sleep((attempt + 1) * 5)
            else:
                raise
    raise RuntimeError("OpenAI gagal")


def openai_embed(texts):
    for attempt in range(5):
        try:
            resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
            return [d.embedding for d in resp.data]
        except Exception as e:
            err = str(e)
            if "429" in err or "rate" in err.lower():
                time.sleep((attempt + 1) * 10)
            elif "500" in err or "502" in err or "503" in err:
                time.sleep((attempt + 1) * 5)
            else:
                raise
    raise RuntimeError("OpenAI embed gagal")


_test = openai_generate("Reply OK", max_tokens=5)
print(f"OpenAI ready: {_test!r}")


OpenAI ready: 'OK'


In [5]:
# ============================================================
# Context Reranking dengan CrossEncoder
# ============================================================
from sentence_transformers import CrossEncoder

print("Loading CrossEncoder ms-marco-MiniLM-L-6-v2...")
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
cross_encoder = CrossEncoder(RERANKER_MODEL)
print("CrossEncoder ready")


def rerank_with_cross_encoder(query, candidates, k_final):
    """Rerank list of RetrievalResult with CrossEncoder."""
    if not candidates:
        return []
    pairs = [(query, r.document.text) for r in candidates]
    scores = cross_encoder.predict(pairs)
    for r, s in zip(candidates, scores):
        r.reranker_score = float(s)
    return sorted(candidates, key=lambda r: -r.reranker_score)[:k_final]


Loading CrossEncoder ms-marco-MiniLM-L-6-v2...


Loading weights: 100%|███████████████████████| 105/105 [00:00<00:00, 484.74it/s, Materializing param=classifier.weight]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CrossEncoder ready


In [6]:
def retrieve_dense(query, k=TOP_K_DENSE):
    qvec = openai_embed([query])[0]
    res = chroma_collection.query(query_embeddings=[qvec], n_results=k, include=["distances"])
    doc_ids = [int(i) for i in res["ids"][0]]
    distances = res["distances"][0]
    scores = [1.0 - d for d in distances]
    return list(zip(doc_ids, scores))


def retrieve_bm25_raw(query, k=TOP_K_BM25):
    tokens = tokenize_bm25(query)
    scores = bm25_index.get_scores(tokens)
    top_k = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top_k]


def reciprocal_rank_fusion(rank_lists, k=60):
    scores = {}
    for rl in rank_lists:
        for rank, doc_id in enumerate(rl):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])


def retrieve_hybrid(query, k_final=TOP_K_RETRIEVAL):
    """Single query Hybrid + CrossEncoder rerank. Get top-20 hybrid, rerank to top-5."""
    bm25_results = retrieve_bm25_raw(query, k=TOP_K_BM25)
    dense_results = retrieve_dense(query, k=TOP_K_DENSE)
    bm25_scores = dict(bm25_results); dense_scores = dict(dense_results)
    fused = reciprocal_rank_fusion([
        [d for d, _ in bm25_results], [d for d, _ in dense_results]
    ])
    # Get top-K_RERANKER candidates
    candidates = []
    for doc_id, rrf in fused[:TOP_K_RERANKER]:
        candidates.append(RetrievalResult(
            document=documents[doc_id], score=rrf, doc_id=doc_id,
            bm25_score=bm25_scores.get(doc_id, 0.0),
            dense_score=dense_scores.get(doc_id, 0.0), rrf_score=rrf,
        ))
    # Rerank with CrossEncoder
    reranked = rerank_with_cross_encoder(query, candidates, k_final=k_final)
    return reranked, query


# Smoke test
test_q = pubmedqa_data[0]["question"]
test_r, _ = retrieve_hybrid(test_q)
print(f"Query: {test_q}")
print(f"\nTop-{TOP_K_RETRIEVAL} chunks (after CrossEncoder rerank from top-{TOP_K_RERANKER}):")
for i, r in enumerate(test_r, 1):
    print(f"  [{i}] CE={r.reranker_score:6.3f} | BM25={r.bm25_score:6.2f} | Dense={r.dense_score:.3f} | "
          f"({r.document.section_label[:25]}) pubid={r.document.pubid}")


Query: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Top-5 chunks (after CrossEncoder rerank from top-20):
  [1] CE= 6.543 | BM25= 61.53 | Dense=0.620 | (BACKGROUND) pubid=21645374
  [2] CE= 4.042 | BM25= 40.85 | Dense=0.675 | (RESULTS) pubid=21645374
  [3] CE=-9.369 | BM25= 16.13 | Dense=0.368 | (KEY RESULTS) pubid=18222909
  [4] CE=-10.043 | BM25= 12.00 | Dense=0.000 | (UNLABELLED) pubid=17279467
  [5] CE=-10.405 | BM25= 13.29 | Dense=0.359 | (BACKGROUND AND AIMS) pubid=18222909


In [7]:
GENERATION_PROMPT = (
    "You are a medical research assistant. "
    "Answer a biomedical yes/no/maybe question based solely on the provided scientific abstracts.\n\n"
    "Context from medical literature:\n{context}\n\n"
    "Question: {question}\n\n"
    "Instructions:\n"
    "- Carefully read the context and assess whether it supports or refutes the question.\n"
    "- Provide a brief explanation (2-3 sentences) using ONLY the information above.\n"
    "- End your response with EXACTLY ONE of these words on its own line: yes, no, or maybe.\n"
    "  - yes   : the evidence supports the hypothesis, even if not perfectly conclusive\n"
    "  - no    : the evidence refutes or does not support the hypothesis\n"
    "  - maybe : ONLY if the evidence is directly contradictory (some findings say yes,\n"
    "            others say no), or if the context contains no relevant information at all\n"
    "- IMPORTANT: If the evidence leans in one direction, even partially, choose yes or no.\n"
    "  Do NOT use maybe simply because the evidence is limited or not 100% certain.\n\n"
    "Answer:"
)


def generate_answer(query, retrieved):
    context = "\n\n".join(
        f"[{i}] ({r.document.section_label}): {r.document.text}"
        for i, r in enumerate(retrieved, 1)
    )
    return openai_generate(
        GENERATION_PROMPT.format(context=context, question=query),
        max_tokens=300, temperature=TEMPERATURE
    )


def extract_label(answer):
    lines = [l.strip().lower() for l in answer.split("\n") if l.strip()]
    for line in reversed(lines[-3:]):
        word = re.sub(r"[^a-z]", "", line)
        if word in ("yes", "no", "maybe"):
            return word
    for label in ("yes", "no", "maybe"):
        if re.search(r"\b" + label + r"\b", answer.lower()):
            return label
    return "maybe"


## Phase 1 — Generate jawaban 500 sampel (resumable, 15 menit)

In [8]:
if PHASE1_PATH.exists():
    with open(PHASE1_PATH, "r", encoding="utf-8") as f:
        results_p1 = json.load(f)["results"]
    start_from = len(results_p1)
    print(f"Resume Fase 1: {start_from}/{MAX_SAMPLES} sudah selesai.")
else:
    results_p1, start_from = [], 0
    print(f"Memulai Fase 1: {MAX_SAMPLES} sampel.")

if start_from < MAX_SAMPLES:
    t0 = time.time()
    for i in range(start_from, MAX_SAMPLES):
        s = pubmedqa_data[i]
        q, gt, ref = s["question"], s["final_decision"], s["long_answer"]

        retrieved, used_q = retrieve_hybrid(q)
        answer = generate_answer(q, retrieved)
        predicted = extract_label(answer)

        results_p1.append({
            "idx": i, "pubid": str(s["pubid"]), "question": q,
            
            "ground_truth": gt, "predicted_label": predicted,
            "is_correct": predicted == gt, "answer": answer,
            "contexts": [r.document.text for r in retrieved],
            "context_pubids": [r.document.pubid for r in retrieved],
            "context_sections": [r.document.section_label for r in retrieved],
            "reference": ref,
            "retrieval_scores": [r.bm25_score for r in retrieved],
            "dense_scores": [r.dense_score for r in retrieved],
            "rrf_scores": [r.rrf_score for r in retrieved],
            "reranker_scores": [r.reranker_score for r in retrieved],
        })

        if (i + 1) % 10 == 0 or i == MAX_SAMPLES - 1:
            with open(PHASE1_PATH, "w", encoding="utf-8") as f:
                json.dump({"config": CONFIG_NAME, "llm_model": LLM_MODEL,
                           "embed_model": EMBED_MODEL,
                           "timestamp": datetime.now().isoformat(),
                           "max_samples": MAX_SAMPLES, "completed": i+1,
                           "results": results_p1}, f, indent=2, ensure_ascii=False)
            done = i + 1
            acc = sum(r["is_correct"] for r in results_p1) / done
            eta = (time.time() - t0) / (i + 1 - start_from) * (MAX_SAMPLES - i - 1) / 60
            print(f"  [{done:3d}/{MAX_SAMPLES}] acc={acc:.1%} | ETA {eta:.1f} mnt")

print(f"\nFase 1 selesai -> {PHASE1_PATH}")


Memulai Fase 1: 500 sampel.
  [ 10/500] acc=60.0% | ETA 22.7 mnt
  [ 20/500] acc=75.0% | ETA 21.1 mnt
  [ 30/500] acc=76.7% | ETA 20.6 mnt
  [ 40/500] acc=75.0% | ETA 19.9 mnt
  [ 50/500] acc=76.0% | ETA 19.4 mnt
  [ 60/500] acc=71.7% | ETA 19.1 mnt
  [ 70/500] acc=74.3% | ETA 18.8 mnt
  [ 80/500] acc=73.8% | ETA 18.6 mnt
  [ 90/500] acc=72.2% | ETA 18.1 mnt
  [100/500] acc=72.0% | ETA 17.9 mnt
  [110/500] acc=73.6% | ETA 17.4 mnt
  [120/500] acc=71.7% | ETA 16.8 mnt
  [130/500] acc=70.0% | ETA 16.3 mnt
  [140/500] acc=69.3% | ETA 15.8 mnt
  [150/500] acc=68.7% | ETA 15.3 mnt
  [160/500] acc=70.0% | ETA 14.9 mnt
  [170/500] acc=71.2% | ETA 14.5 mnt
  [180/500] acc=71.7% | ETA 14.0 mnt
  [190/500] acc=71.6% | ETA 13.5 mnt
  [200/500] acc=72.0% | ETA 13.1 mnt
  [210/500] acc=71.9% | ETA 12.6 mnt
  [220/500] acc=71.4% | ETA 12.2 mnt
  [230/500] acc=71.3% | ETA 11.8 mnt
  [240/500] acc=70.8% | ETA 11.4 mnt
  [250/500] acc=70.8% | ETA 10.9 mnt
  [260/500] acc=71.2% | ETA 10.4 mnt
  [270/500

## Phase 1 — Quick analysis

In [5]:
with open(PHASE1_PATH, "r", encoding="utf-8") as f:
    results_p1 = json.load(f)["results"]

n = len(results_p1)
n_correct = sum(r["is_correct"] for r in results_p1)
print(f"Schwartz-Hearst + CR ({n} sampel)")
print("=" * 60)
print(f"Label Accuracy : {n_correct}/{n} = {n_correct/n:.1%}")
print()
print("Per-label accuracy:")
for lbl in ["yes", "no", "maybe"]:
    sub = [r for r in results_p1 if r["ground_truth"] == lbl]
    if sub:
        c = sum(r["is_correct"] for r in sub)
        print(f"  {lbl:>5}: {c}/{len(sub)} = {c/len(sub):.1%}")

# Comparison with other SH variants
print(f"\n=== COMPARISON dalam SH family ===")
configs = {
    "SH baseline       ": RESULTS_DIR / "sh_baseline_openai_phase1_answers.json",
    "SH + QR           ": RESULTS_DIR / "sh_qr_openai_phase1_answers.json",
    "SH + CR           ": RESULTS_DIR / "sh_cr_openai_phase1_answers.json",
    "SH + QR + CR      ": RESULTS_DIR / "sh_qr_cr_openai_phase1_answers.json",
}
for name, path in configs.items():
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            d = json.load(f)["results"]
        a = sum(r["is_correct"] for r in d) / len(d)
        marker = "  <-- THIS" if path == PHASE1_PATH else ""
        print(f"  {name}: {a:.1%}{marker}")


Schwartz-Hearst + CR (500 sampel)
Label Accuracy : 367/500 = 73.4%

Per-label accuracy:
    yes: 247/275 = 89.8%
     no: 116/159 = 73.0%
  maybe: 4/66 = 6.1%

=== COMPARISON dalam SH family ===
  SH baseline       : 73.2%
  SH + QR           : 73.0%
  SH + CR           : 73.4%  <-- THIS
  SH + QR + CR      : 74.8%


## Phase 2 — Custom evaluator 4 metrik (resumable, 25 menit)

In [6]:
def _split_sentences(text):
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s.strip() for s in parts if len(s.strip()) >= 15]


def _llm_yes_no(prompt):
    try:
        resp = openai_generate(prompt, max_tokens=10, temperature=0.0)
        return "yes" in resp.lower()[:15]
    except Exception:
        return False


def compute_faithfulness(answer, contexts):
    sents = _split_sentences(answer)
    if not sents: return 0.0
    ctx_text = "\n".join(f"[{i+1}] {c[:1500]}" for i, c in enumerate(contexts))
    tmpl = ("Context:\n{ctx}\n\nStatement: {sent}\n\n"
            "Is this statement directly supported by the context above? "
            "Answer with only \"yes\" or \"no\".")
    return sum(_llm_yes_no(tmpl.format(ctx=ctx_text, sent=s)) for s in sents) / len(sents)


def compute_context_recall(reference, contexts):
    sents = _split_sentences(reference)
    if not sents: return 0.0
    ctx_text = "\n".join(f"[{i+1}] {c[:1500]}" for i, c in enumerate(contexts))
    tmpl = ("Context:\n{ctx}\n\nStatement: {sent}\n\n"
            "Is this statement supported by the context above? "
            "Answer with only \"yes\" or \"no\".")
    return sum(_llm_yes_no(tmpl.format(ctx=ctx_text, sent=s)) for s in sents) / len(sents)


def compute_answer_relevancy(question, answer):
    sents = _split_sentences(answer)
    if not sents: return 0.0
    tmpl = ("Question: {q}\n\nStatement: {sent}\n\n"
            "Is this statement relevant to answering the question above? "
            "Answer with only \"yes\" or \"no\".")
    return sum(_llm_yes_no(tmpl.format(q=question, sent=s)) for s in sents) / len(sents)


def compute_context_precision(question, contexts, reference):
    if not contexts: return 0.0
    tmpl = ("Question: {q}\n\nGround truth answer: {ref}\n\nRetrieved context: {ctx}\n\n"
            "Does this context contain information useful for correctly answering "
            "the question based on the ground truth? Answer with only \"yes\" or \"no\".")
    relevance = [1 if _llm_yes_no(tmpl.format(q=question, ref=reference[:800], ctx=c[:1500])) else 0
                 for c in contexts]
    total = sum(relevance)
    if total == 0: return 0.0
    prec_sum = 0.0; rel_count = 0
    for k, rel in enumerate(relevance):
        if rel:
            rel_count += 1; prec_sum += rel_count / (k + 1)
    return prec_sum / total


def evaluate_custom(q, ans, ctx, ref):
    return {
        "faithfulness": compute_faithfulness(ans, ctx),
        "context_recall": compute_context_recall(ref, ctx),
        "answer_relevancy": compute_answer_relevancy(q, ans),
        "context_precision": compute_context_precision(q, ctx, ref),
    }


print("Evaluator ready.")


Evaluator ready.


In [7]:
REQ = ["faithfulness", "context_recall", "answer_relevancy", "context_precision"]

with open(PHASE1_PATH, "r", encoding="utf-8") as f:
    p1_results = json.load(f)["results"][:MAX_SAMPLES]

if PHASE2_PATH.exists():
    with open(PHASE2_PATH, "r", encoding="utf-8") as f:
        p2_results = json.load(f)["results"]
    done = {r["idx"] for r in p2_results if all(m in r for m in REQ)}
else:
    p2_results, done = [], set()

remaining = [r for r in p1_results if r["idx"] not in done]
print(f"Phase 2: {len(done)}/{MAX_SAMPLES} done, {len(remaining)} remaining.")

if remaining:
    t0 = time.time()
    for i, r in enumerate(remaining):
        scores = evaluate_custom(r["question"], r["answer"], r["contexts"], r["reference"])
        p2_results.append({
            "idx": r["idx"], "ground_truth": r["ground_truth"],
            "predicted_label": r["predicted_label"], "is_correct": r["is_correct"],
            **scores,
        })
        if (i + 1) % 10 == 0 or i == len(remaining) - 1:
            with open(PHASE2_PATH, "w", encoding="utf-8") as f:
                json.dump({"config": CONFIG_NAME,
                           "timestamp": datetime.now().isoformat(),
                           "max_samples": MAX_SAMPLES,
                           "metrics": REQ, "evaluator": "custom_zero_nan_4metrics",
                           "results": p2_results}, f, indent=2, ensure_ascii=False)
            done_n = i + 1
            avg = {m: sum(x[m] for x in p2_results) / len(p2_results) for m in REQ}
            eta = (time.time() - t0) / (i + 1) * (len(remaining) - i - 1) / 60
            print(f"  [{done_n:3d}/{len(remaining)}] "
                  f"f={avg['faithfulness']:.3f} cr={avg['context_recall']:.3f} "
                  f"ar={avg['answer_relevancy']:.3f} cp={avg['context_precision']:.3f} | ETA {eta:.1f}m")

print(f"\nFase 2 selesai -> {PHASE2_PATH}")


Phase 2: 0/500 done, 500 remaining.
  [ 10/500] f=1.000 cr=0.867 ar=0.950 cp=0.842 | ETA 71.4m
  [ 20/500] f=0.975 cr=0.867 ar=0.958 cp=0.854 | ETA 76.3m
  [ 30/500] f=0.983 cr=0.911 ar=0.956 cp=0.833 | ETA 76.3m
  [ 40/500] f=0.988 cr=0.933 ar=0.967 cp=0.790 | ETA 71.3m
  [ 50/500] f=0.990 cr=0.927 ar=0.973 cp=0.786 | ETA 74.6m
  [ 60/500] f=0.992 cr=0.911 ar=0.978 cp=0.794 | ETA 72.9m
  [ 70/500] f=0.993 cr=0.917 ar=0.976 cp=0.801 | ETA 69.7m
  [ 80/500] f=0.994 cr=0.909 ar=0.979 cp=0.776 | ETA 68.7m
  [ 90/500] f=0.994 cr=0.895 ar=0.974 cp=0.766 | ETA 67.0m
  [100/500] f=0.992 cr=0.889 ar=0.973 cp=0.769 | ETA 65.0m
  [110/500] f=0.989 cr=0.891 ar=0.970 cp=0.757 | ETA 62.6m
  [120/500] f=0.986 cr=0.880 ar=0.968 cp=0.755 | ETA 60.4m
  [130/500] f=0.985 cr=0.889 ar=0.971 cp=0.763 | ETA 58.0m
  [140/500] f=0.986 cr=0.892 ar=0.973 cp=0.774 | ETA 56.0m
  [150/500] f=0.987 cr=0.894 ar=0.974 cp=0.772 | ETA 54.9m
  [160/500] f=0.984 cr=0.895 ar=0.976 cp=0.774 | ETA 53.4m
  [170/500] f=0.985 